# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. We will walk through data loading, overview, extraction, exploratory analysis, and simple visualization.

### Dataset Source
The dataset is described by a Croissant schema at:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object (not as a dict or list)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published on: {metadata.datePublished}")

## 2. Data Overview
Review the available record sets, fields, and their `@id` fields. You can list all record set IDs, then view their fields and columns.

**Note**: All entities are referenced by their `@id`.

In [ ]:
# List all record set IDs available in the dataset
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
print("Record set @id's in this dataset:")
for rid in record_set_ids:
    print(f"  - {rid}")

print("\n-- Fields in each record set --")
for rid in record_set_ids:
    rs = dataset.metadata.recordSet(rid)
    field_ids = [fld['@id'] for fld in rs.fields]
    print(f"Record set: {rid}\n  Fields: {field_ids}\n")

# (Optional): Preview the first record in each record set
for rid in record_set_ids:
    print(f'First row from record set {rid}:')
    for rec in dataset.records(record_set=rid):
        print(rec)
        break

## 3. Data Extraction
We'll load data from all record sets into pandas DataFrames for analysis. All references use the record set and field `@id` values obtained previously.

Feel free to pick which record set to explore further for later steps.

In [ ]:
# Load records from all record sets into DataFrames
dataframes = {}
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    df = pd.DataFrame(records)
    dataframes[rid] = df

# Choose the first record set as the example to explore
if record_set_ids:
    main_rid = record_set_ids[0]
    print(f"Fields (DataFrame columns) for record set '{main_rid}':")
    print(dataframes[main_rid].columns.tolist())
    print("\nSample records:")
    display(dataframes[main_rid].head())
else:
    print("No record sets are defined in this dataset's Croissant schema.")

## 4. Exploratory Data Analysis (EDA)
Now let's process the loaded data:
- Filter on a numeric field (e.g., any coefficient, score, count, etc.)
- Remove outliers, normalize, or group by a categorical field as appropriate.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` with actual `@id`s present in the selected record set.

In [ ]:
# Define which record set and fields to process (customize as needed)
from IPython.display import display

selected_rs_id = main_rid  # Use the main record set from previous cell
df = dataframes[selected_rs_id]
print(f"Available fields (@id) for EDA: {list(df.columns)}")

# Choose a numeric field and a group/categorical field (edit as appropriate)
numeric_field = None
group_field = None
for col in df.columns:
    # Heuristic: Pick a field containing 'coef' or 'count' or is float/int dtype
    if numeric_field is None and (
        'coef' in col.lower() or 'count' in col.lower() or pd.api.types.is_numeric_dtype(df[col])
    ):
        numeric_field = col
    # Heuristic: Pick a likely grouping field
    if group_field is None and ('ward' in col.lower() or 'gender' in col.lower() or pd.api.types.is_object_dtype(df[col])):
        group_field = col

# Print our field choices
print(f"\nUsing numeric field: {numeric_field}")
print(f"Using group field: {group_field}")

if numeric_field and numeric_field in df.columns:
    # Drop missing values
    filtered_df = df[df[numeric_field].notnull()].copy()

    # Example: Filter values above their mean
    threshold = filtered_df[numeric_field].mean()
    high_df = filtered_df[filtered_df[numeric_field] > threshold].copy()
    print(f"\n{len(high_df)} records with {numeric_field} above mean of {threshold:.2f}")
    display(high_df[[numeric_field]].head())

    # Normalize the numeric field
    colname = f"{numeric_field}_normalized"
    high_df[colname] = (high_df[numeric_field] - high_df[numeric_field].mean()) / high_df[numeric_field].std()
    print("\nNormalized numeric field (first records):")
    display(high_df[[numeric_field, colname]].head())

    # Group by the group_field if present
    if group_field and group_field in high_df.columns:
        group_stats = high_df.groupby(group_field)[numeric_field].agg(['mean', 'count'])
        print(f"\nGrouped stats by '{group_field}':")
        display(group_stats.head())
else:
    print("No numeric field detected in the selected DataFrame. Please revise field selection.")

## 5. Visualization
Let's visualize the distribution of our chosen numeric field and/or compare groups.

Visualization adjusts automatically to available fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=25, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # If group_field was found, add group comparison
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
This notebook explored the FAIR² dataset using the Croissant schema and the `mlcroissant` library. We loaded record sets by `@id`, examined fields, performed simple EDA (filtering and normalization), and visualized key data attributes.

**Next steps:** Deeper domain-specific analyses can be performed based on the research questions and variable descriptions provided in the dataset metadata.